In [ ]:
from __future__ import annotations
import json
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Mapping, Sequence, Tuple
import numpy as np
import pandas as pd


@dataclass(frozen=True)
class GelatoPaths:
    nutrients_pred_csv: Path
    phenotype_csv: Path
    medium_json: Path
    standard_name_json: Path
    species2medium_json: Path
    cluster_medium_csv: Path
    out_dir: Path


P = GelatoPaths(
    nutrients_pred_csv=Path(
        #"Medium components possibility.csv"
        "/Kun_data/gene_language_model/recommend_medium/gelato/Medium components possibility.csv"
    ),
    phenotype_csv=Path("../data/phenotypes.csv"),
    medium_json=Path("../data/medium.json"),
    standard_name_json=Path("../data/standard_name.json"),
    species2medium_json=Path("../data/species2medium.json"),
    cluster_medium_csv=Path("../data/cluster_medium.csv"),
    out_dir=Path("gelato_pred"),
)

TOP_K = 50
LAMBDA_PENALTY = 0.015

P.out_dir.mkdir(parents=True, exist_ok=True)


def to_DSMZ(name: str) -> str:
    m = re.search(r"\(DSMZ Medium ([0-9a-z.]+)\)", name)
    if m:
        return f"DSMZ_Medium{m.group(1)}"
    return name


def load_json(path: Path):
    with open(path) as f:
        return json.load(f)


def build_compounds_list(phenotype_df: pd.DataFrame) -> List[str]:
    remove = {"Malachite green cation", "Resazurin", "EDTA", "Nitrobenzol", "Catalase"}
    compounds = phenotype_df[phenotype_df["Category"] == "Microbial growth medium components"]["Object"].tolist()
    return [c for c in compounds if c not in remove]


def build_medium_vector_df(
    *,
    medium_json: Mapping,
    standard_name_json: Mapping[str, Sequence[str]],
    compounds_ls: Sequence[str],
) -> pd.DataFrame:
    standard_name_unique: Dict[str, List[str]] = {
        k: sorted(set(v)) for k, v in standard_name_json.items()
    }

    # medium -> set(compounds)
    split_medium: Dict[str, List[str]] = {}
    for medium_name, org_dict in medium_json.items():
        org_ls = list(org_dict.keys())
        expanded: List[str] = []
        for org in org_ls:
            org = str(org).strip()
            if org in standard_name_unique:
                expanded.extend(standard_name_unique[org])
        split_medium[medium_name] = sorted(set(expanded))

    compound2idx = {c: i for i, c in enumerate(compounds_ls)}
    rows: List[List[int]] = []
    names: List[str] = []
    for m, comps in split_medium.items():
        vec = [0] * len(compounds_ls)
        hit = False
        for c in comps:
            if c in compound2idx:
                vec[compound2idx[c]] = 1
                hit = True
        if hit:
            names.append(m)
            rows.append(vec)

    df = pd.DataFrame(rows, columns=list(compounds_ls))
    df.insert(0, "Medium", names)
    df["num_of_compounds"] = df[list(compounds_ls)].sum(axis=1)
    df = df[df["num_of_compounds"] > 0].reset_index(drop=True)
    return df


def pointbiserial_matrix(y: np.ndarray, x_bin: np.ndarray) -> np.ndarray:
    """Vectorized point-biserial corr.

    y: (n_species, n_compounds) continuous in [0,1]
    x_bin: (n_medium, n_compounds) binary {0,1}
    return: (n_species, n_medium)

    r = (mean1 - mean0) / s_y * sqrt(p*q)
    """
    y = np.asarray(y, dtype=np.float64)
    x = np.asarray(x_bin, dtype=np.float64)

    n = y.shape[1]
    if x.shape[1] != n:
        raise ValueError("y and x must share n_compounds")

    # p for each medium
    p = x.mean(axis=1)  # (n_medium,)
    q = 1.0 - p

    sum1 = y @ x.T
    sum0 = y @ (1.0 - x).T

    n1 = np.clip((p * n), 1e-12, None)
    n0 = np.clip((q * n), 1e-12, None)
    mean1 = sum1 / n1
    mean0 = sum0 / n0

    # std of y per species
    y_std = y.std(axis=1, ddof=1)
    y_std = np.where(y_std == 0, np.nan, y_std)

    r = (mean1 - mean0) / y_std[:, None] * np.sqrt(p * q)[None, :]
    return r


In [ ]:
nutrients_pred_raw = pd.read_csv(P.nutrients_pred_csv)
phenotype_df = pd.read_csv(P.phenotype_csv)

compounds_ls = build_compounds_list(phenotype_df)

medium_json = load_json(P.medium_json)
standard_name_json = load_json(P.standard_name_json)

medium_vec_df = build_medium_vector_df(
    medium_json=medium_json,
    standard_name_json=standard_name_json,
    compounds_ls=compounds_ls,
)


keep_compounds = [c for c in compounds_ls if c in nutrients_pred_raw.columns]

nutrients_pred = nutrients_pred_raw[keep_compounds + ["Query"]].copy()

print("species n:", len(nutrients_pred))
print("medium n:", len(medium_vec_df))
print("compounds n (kept):", len(keep_compounds))

nutrients_pred.head(3)

species n: 10623
medium n: 2439
compounds n (kept): 70


,Agar,Aminobenzoic acid,BH3O3,Beef extract,Biotinate,Br(-),Ca(2+),Casamino acids,Casein hydrolysate,Casitone,...,Tryptone,Tryptone Soy Broth,Tryptone casein soy agar,Tween,Vitamin Mix,WO4(2-),Yeast Extract,Zn(2+),brain_heart,Query
0,0.580329,0.361893,0.140466,0.202068,0.299312,0.211163,0.320545,0.514807,0.119387,0.217648,...,0.118347,0.061484,0.094494,0.145331,0.438694,0.146873,0.570482,0.186056,0.230606,Abditibacterium utsteinense
1,0.355269,0.352582,0.007301,0.808322,0.166834,0.054676,0.258968,0.032243,0.258937,0.161818,...,0.742047,0.488490,0.428987,0.882188,0.401088,0.117280,0.860295,0.002070,0.799898,Abiotrophia defectiva
2,0.178139,0.436497,0.015506,0.840060,0.354375,0.022675,0.302742,0.036709,0.111427,0.579765,...,0.641131,0.256045,0.117054,0.681008,0.582760,0.309574,0.358082,0.035128,0.730286,Absicoccus porci


In [3]:
species_names = nutrients_pred["Query"].astype(str).to_numpy()
medium_names = medium_vec_df["Medium"].astype(str).to_numpy()

species_matrix = nutrients_pred[keep_compounds].to_numpy(dtype=np.float64)          # (S, C)
medium_matrix = medium_vec_df[keep_compounds].to_numpy(dtype=np.int8)              # (M, C)
num_compounds = medium_vec_df["num_of_compounds"].to_numpy(dtype=np.float64)       # (M,)

cache_path = P.out_dir / f"pbc_corr_S{species_matrix.shape[0]}_M{medium_matrix.shape[0]}_C{species_matrix.shape[1]}.npy"

if cache_path.exists():
    corr = np.load(cache_path)
else:
    corr = pointbiserial_matrix(species_matrix, medium_matrix)
    np.save(cache_path, corr)

print("corr shape:", corr.shape)  # (S, M)


score = corr - (LAMBDA_PENALTY * num_compounds[None, :])
score = np.where(np.isfinite(score) & (score > 0), score, -np.inf)

# topK indices per species
k = min(TOP_K, score.shape[1])
idx_part = np.argpartition(-score, kth=k - 1, axis=1)[:, :k]
row_ids = np.arange(score.shape[0])[:, None]
score_top = score[row_ids, idx_part]
order = np.argsort(-score_top, axis=1)
idx_top = idx_part[row_ids, order]

pred_medium_topk = [[medium_names[j] for j in idx_top[i] if np.isfinite(score[i, j])] for i in range(score.shape[0])]
out_topk = pd.DataFrame({"Species": species_names, f"Pred_medium_top{TOP_K}": pred_medium_topk})
out_topk_path = P.out_dir / f"gelato_pred_top{TOP_K}.csv"
out_topk.to_csv(out_topk_path, index=False)

corr shape: (10623, 2439)


In [4]:
species2medium = load_json(P.species2medium_json)
cluster_medium_df = pd.read_csv(P.cluster_medium_csv)

m2sc = dict(zip(cluster_medium_df["Medium"].astype(str), cluster_medium_df["small_cluster_id"]))

true_medium_lst = []
pred_medium_lst = []
true_cluster_lst = []
pred_cluster_lst = []

for i, sp in enumerate(species_names):
    tm = species2medium.get(sp, [])
    pm = pred_medium_topk[i]

    true_medium_lst.append(tm)
    pred_medium_lst.append(pm)
    true_cluster_lst.append([m2sc.get(m, -1) for m in tm])
    pred_cluster_lst.append([m2sc.get(m, -1) for m in pm])

pred_df = pd.DataFrame(
    {
        "Species": species_names,
        "True_medium": true_medium_lst,
        "Pred_medium": pred_medium_lst,
        "True_cluster": true_cluster_lst,
        "Pred_cluster": pred_cluster_lst,
    }
)

pred_df.head(3)

,Species,True_medium,Pred_medium,True_cluster,Pred_cluster
0,Abditibacterium utsteinense,[DSMZ_Medium830],"[DSMZ_Medium1153, DSMZ_Medium1320, DSMZ_Medium...",[3],"[3, 3, 3, 86, 86, 86, 86, 86, 88, 39, 70, 4, 4..."
1,Abiotrophia defectiva,"[DSMZ_Medium104, MEDIUM 27 - for Capnocytophag...","[MEDIUM 40- for Lactobacillus and Leuconostoc,...","[42, 24, 24, 22, 22, 22]","[41, 42, 71, 2, 2, 2, 65, 2, 2, 2, 2, 2, 2, 2,..."
2,Absicoccus porci,[brain heart infusion agar with blood],"[MEDIUM 40- for Lactobacillus and Leuconostoc,...",[-1],"[41, 14, 26, 18, 73, 73, 14, 14, 16, 16, 16, 4..."


In [5]:
out_pred_path = P.out_dir / "gelato_pred.csv"
pred_df.to_csv(out_pred_path, index=False)

In [6]:
def intersection_count(list1, list2) -> int:
    return len(set(list1) & set(list2))


def cluster_hit_acc(df: pd.DataFrame, top_k: int = 10) -> Tuple[int, int, float]:
    correct = 0


    species_col = df["Species"]
    if isinstance(species_col, pd.DataFrame):
        species_series = species_col.iloc[:, 0]
    else:
        species_series = species_col

    total = int(species_series.nunique())

    for _, row in df.iterrows():
        pred_clusters = row["Pred_cluster"][:top_k]
        true_clusters = row["True_cluster"]
        if intersection_count(pred_clusters, true_clusters) > 0:
            correct += 1

    acc = correct / total if total else 0.0
    return correct, total, acc


correct, total, acc = cluster_hit_acc(pred_df, top_k=10)
print(f"Correct: {correct}, Total: {total}")
print(f"Accuracy@10: {acc:.5f}")


Correct: 7974, Total: 10623
Accuracy@10: 0.75064


In [7]:
import ast

fastani_path = Path("/Kun_data/gene_language_model/recommend_medium/ani_pred_result/fastani_env_result.csv")
fastani_df = pd.read_csv(fastani_path)

fastani_df = fastani_df.copy()
fastani_df["score"] = fastani_df["ANI"] + fastani_df["Env_similarity"] * 100
fastani_df['True_clusters'] = fastani_df['True_clusters'].apply(ast.literal_eval)
fastani_df['Pred_clusters'] = fastani_df['Pred_clusters'].apply(ast.literal_eval)

In [8]:
def has_overlap(list1, list2):
    return 1 if set(list1) & set(list2) else 0

def ani_env_recommend(query_list, ani_df, top_k):
    grouped = {q: g for q, g in ani_df.groupby("Query_species", sort=False)}

    correct_ani = 0
    correct_ani_env = 0
    total = len(query_list)

    if total == 0:
        return 0.0, 0.0, 0.0

    for q in query_list:
        sub_df = grouped.get(q)
        if sub_df is None or sub_df.empty:
            continue

        true_cluster_lst = sub_df['True_clusters'].iloc[0]

        ani_pred_df = sub_df.sort_values(by="ANI", ascending=False)
        ani_env_pred_df = sub_df.sort_values(by="score", ascending=False)

        ani_pred_lst = []
        for ls in ani_pred_df["Pred_clusters"].tolist():
            ani_pred_lst.extend(ls)
        ani_pred_lst = ani_pred_lst[:top_k]
        ani_env_pred_lst = []
        
        for ls in ani_env_pred_df["Pred_clusters"].tolist():
            ani_env_pred_lst.extend(ls)
        ani_env_pred_lst = ani_env_pred_lst[:top_k]
        
        ani_pred_result = has_overlap(true_cluster_lst, ani_pred_lst)
        ani_env_pred_result = has_overlap(true_cluster_lst, ani_env_pred_lst)

        correct_ani += ani_pred_result
        correct_ani_env += ani_env_pred_result
        
    acc_ani = correct_ani / total
    acc_ani_env = correct_ani_env / total

    return acc_ani, acc_ani_env, total


intervals = [
    # (0, 74),
    # (74, 76),
    # (76, 78),
    # (78, 80),
    # (80, 82),
    # (82, 84),
    # (84, 86),
    # (86, 88),
    # (88, 90),
    # (90, 92),
    # (92, 94),
    # (94, 96),
    # (96, 98),
    # (98, 100)

    (0, 74),
    (74, 78),
    (76, 80),
    (78, 82),
    (80, 84),
    (82, 86),
    (84, 88),
    (86, 90),
    (88, 92),
    (90, 94),
    (92, 96),
    (94, 98),
    (96, 100)
]

topk_lst = [5,10,15,20]
for top_k in topk_lst:
    ani_result = []
    ani_env_result = []
    total_lst = []
    for lo, hi in intervals:
        if hi == 100:
            sub_fastani = fastani_df[(fastani_df["ANI"] >= lo) & (fastani_df["ANI"] <= hi)]
        else:
            sub_fastani = fastani_df[(fastani_df["ANI"] >= lo) & (fastani_df["ANI"] < hi)]

        species_in_bin = sub_fastani["Query_species"].drop_duplicates().tolist()
        acc_ani, acc_ani_env, total = ani_env_recommend(species_in_bin, sub_fastani, top_k)

        ani_result.append(acc_ani)
        ani_env_result.append(acc_ani_env)
        total_lst.append(total)
    

    df = pd.DataFrame(
        {
            "ANI": [f"{lo}-{hi}" for lo, hi in intervals],
            "ANI_Acc": ani_result,
            "ANI_ENV_Acc": ani_env_result,
            "Total": total_lst
        }
    )

    out_path = P.out_dir / f"ani_env_top{top_k}_acc.csv"
    df.to_csv(out_path, index=False)


df

,ANI,ANI_Acc,ANI_ENV_Acc,Total
0,0-74,0.000000,0.000000,0.0
1,74-78,0.857093,0.849663,5787.0
2,76-80,0.884274,0.885841,7656.0
3,78-82,0.891338,0.891730,7666.0
4,80-84,0.874173,0.877976,6048.0
5,82-86,0.866638,0.867068,4649.0
6,84-88,0.856660,0.856942,3551.0
7,86-90,0.859827,0.859451,2661.0
8,88-92,0.858943,0.858943,2063.0
9,90-94,0.842074,0.842676,1659.0


In [9]:
def interval_acc_by_ani(
    *,
    fastani_df: pd.DataFrame,
    gelato_pred_df: pd.DataFrame,
    top_k: int = 20,
) -> pd.DataFrame:
    if "Query_species" not in fastani_df.columns or "ANI" not in fastani_df.columns:
        raise KeyError("fastani_df must contain columns: Query_species, ANI")

    intervals: List[Tuple[int, int]] = [
    # (0, 74),
    # (74, 76),
    # (76, 78),
    # (78, 80),
    # (80, 82),
    # (82, 84),
    # (84, 86),
    # (86, 88),
    # (88, 90),
    # (90, 92),
    # (92, 94),
    # (94, 96),
    # (96, 98),
    # (98, 100)
        (0, 74),
        (74, 78),
        (76, 80),
        (78, 82),
        (80, 84),
        (82, 86),
        (84, 88),
        (86, 90),
        (88, 92),
        (90, 94),
        (92, 96),
        (94, 98),
        (96, 100)
    ]

    acc_lst: List[float] = []
    total_lst: List[int] = []

    for lo, hi in intervals:
        if hi == 100:
            sub_fastani = fastani_df[(fastani_df["ANI"] >= lo) & (fastani_df["ANI"] <= hi)]
        else:
            sub_fastani = fastani_df[(fastani_df["ANI"] >= lo) & (fastani_df["ANI"] < hi)]

        species_in_bin = sub_fastani["Query_species"].unique().tolist()
        if len(species_in_bin) == 0:
            all_fastani_species = fastani_df["Query_species"].tolist()
            sub_gelato = gelato_pred_df[~gelato_pred_df["Species"].isin(all_fastani_species)]
        else:
            sub_gelato = gelato_pred_df[gelato_pred_df["Species"].isin(species_in_bin)]


        if len(sub_gelato) > 0:
            correct, total, acc = cluster_hit_acc(sub_gelato, top_k=top_k)
        else:
            correct, total, acc = 0, 0, 0.0

        acc_lst.append(acc)
        total_lst.append(total)

    return pd.DataFrame(
        {
            "ANI": [f"{lo}-{hi}" for lo, hi in intervals],
            "Total": total_lst,
            "GELATO_Acc": acc_lst,
        }
    )

topk_lst = [5,10,15,20]
for top_k in topk_lst:

    df = interval_acc_by_ani(
        fastani_df=fastani_df,
        gelato_pred_df=pred_df,
        top_k=top_k,
    )
    out_path = P.out_dir / f"gelato_top{top_k}_acc.csv"
    df.to_csv(out_path, index=False)

df

,ANI,Total,GELATO_Acc
0,0-74,2185,0.651259
1,74-78,5787,0.842060
2,76-80,7656,0.840648
3,78-82,7666,0.848682
4,80-84,6048,0.865906
5,82-86,4649,0.875457
6,84-88,3551,0.885103
7,86-90,2661,0.885757
8,88-92,2063,0.885119
9,90-94,1659,0.884268


In [10]:
def parse_list(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        return ast.literal_eval(x)
    return list(x)


def has_overlap(list1, list2):
    return 1 if set(list1) & set(list2) else 0


def merge_dict(dict1, dict2):
    merged = dict1.copy()
    for key, value in dict2.items():
        if key in merged:
            merged[key] += value
        else:
            merged[key] = value
    return merged


def min_max_normalize(d, new_min=0.0, new_max=1.0):
    if not d:
        return {}

    values = list(d.values())
    if len(set(values)) == 1:
        return {k: new_min for k in d}

    min_val = min(values)
    max_val = max(values)
    return {
        k: (v - min_val) / (max_val - min_val) * (new_max - new_min) + new_min
        for k, v in d.items()
    }


species_medium_score_dict: Dict[str, Dict[str, float]] = {}

for i, sp in enumerate(species_names):
    row_score = score[i]
    valid_idx = np.where(np.isfinite(row_score) & (row_score > 0))[0]

    medium_score_map = {}
    for j in valid_idx:
        medium = str(medium_names[j])
        medium_score_map[medium] = float(row_score[j])

    species_medium_score_dict[str(sp)] = medium_score_map


species2medium = load_json(P.species2medium_json)
species2medium = {str(k): [str(x) for x in v] for k, v in species2medium.items()}


def recommend_from_result_dict(result_dict, m2sc, top_k):
    if not result_dict:
        return []

    result_dict = min_max_normalize(result_dict)
    sorted_items = sorted(result_dict.items(), key=lambda item: item[1], reverse=True)
    pred_medium_topk = [m for m, s in sorted_items[:top_k]]
    pred_cluster_topk = [m2sc.get(m, -1) for m in pred_medium_topk]
    pred_cluster_topk = [int(x) for x in pred_cluster_topk if x != -1]
    return pred_cluster_topk


def gelato_only_recommend(species_list, gelato_pred_df, species_medium_score_dict, m2sc, top_k):
    correct = 0
    total = len(species_list)

    if total == 0:
        return 0.0, 0

    grouped = {q: g for q, g in gelato_pred_df.groupby("Species", sort=False)}

    for sp in species_list:
        sub_df = grouped.get(sp)
        if sub_df is None or sub_df.empty:
            continue

        true_cluster_lst = [int(x) for x in parse_list(sub_df["True_cluster"].iloc[0])]
        pred_dict = species_medium_score_dict.get(str(sp), {}).copy()

        pred_cluster_topk = recommend_from_result_dict(pred_dict, m2sc, top_k)
        correct += has_overlap(true_cluster_lst, pred_cluster_topk)

    acc = correct / total
    return acc, total


def ani_gelato_recommend_aligned(query_list, ani_df, species_medium_score_dict, species2medium, m2sc, top_k):
    grouped = {q: g for q, g in ani_df.groupby("Query_species", sort=False)}

    correct = 0
    total = len(query_list)

    if total == 0:
        return 0.0, 0

    for q in query_list:
        sub_df = grouped.get(q)
        if sub_df is None or sub_df.empty:
            continue

        true_cluster_lst = [int(x) for x in parse_list(sub_df["True_clusters"].iloc[0])]
        pred_dict = species_medium_score_dict.get(str(q), {}).copy()

        ani_medium = []
        ani_ls = []

        for row in sub_df.itertuples(index=False):
            ref_species = str(row.Ref_species)
            ani_val = float(row.ANI) * 0.01
            ref_mediums = species2medium.get(ref_species, [])

            ani_medium.extend(ref_mediums)
            ani_ls.extend([ani_val] * len(ref_mediums))

        ani_pred_dict = dict(zip(ani_medium, ani_ls))

        result = merge_dict(pred_dict, ani_pred_dict)
        pred_cluster_topk = recommend_from_result_dict(result, m2sc, top_k)

        correct += has_overlap(true_cluster_lst, pred_cluster_topk)

    acc = correct / total
    return acc, total


intervals = [
    # (0, 74),
    # (74, 76),
    # (76, 78),
    # (78, 80),
    # (80, 82),
    # (82, 84),
    # (84, 86),
    # (86, 88),
    # (88, 90),
    # (90, 92),
    # (92, 94),
    # (94, 96),
    # (96, 98),
    # (98, 100),
        (0, 74),
        (74, 78),
        (76, 80),
        (78, 82),
        (80, 84),
        (82, 86),
        (84, 88),
        (86, 90),
        (88, 92),
        (90, 94),
        (92, 96),
        (94, 98),
        (96, 100)
]

topk_lst = [5, 10, 15, 20]

all_fastani_species = set(fastani_df["Query_species"].astype(str).tolist())

for top_k in topk_lst:
    acc_result = []
    total_lst = []

    for lo, hi in intervals:
        if lo == 0 and hi == 74:
            species_in_bin = pred_df.loc[
                ~pred_df["Species"].astype(str).isin(all_fastani_species),
                "Species"
            ].astype(str).tolist()

            acc, total = gelato_only_recommend(
                species_list=species_in_bin,
                gelato_pred_df=pred_df,
                species_medium_score_dict=species_medium_score_dict,
                m2sc=m2sc,
                top_k=top_k,
            )
        else:
            if hi == 100:
                sub_fastani = fastani_df[(fastani_df["ANI"] >= lo) & (fastani_df["ANI"] <= hi)].copy()
            else:
                sub_fastani = fastani_df[(fastani_df["ANI"] >= lo) & (fastani_df["ANI"] < hi)].copy()

            species_in_bin = sub_fastani["Query_species"].astype(str).drop_duplicates().tolist()

            acc, total = ani_gelato_recommend_aligned(
                query_list=species_in_bin,
                ani_df=sub_fastani,
                species_medium_score_dict=species_medium_score_dict,
                species2medium=species2medium,
                m2sc=m2sc,
                top_k=top_k,
            )

        acc_result.append(acc)
        total_lst.append(total)

    df = pd.DataFrame(
        {
            "ANI": [f"{lo}-{hi}" for lo, hi in intervals],
            "ANI_GELATO_Acc": acc_result,
            "Total": total_lst,
        }
    )

    out_path = P.out_dir / f"ani_gelato_top{top_k}_acc.csv"
    df.to_csv(out_path, index=False)

df


,ANI,ANI_GELATO_Acc,Total
0,0-74,0.651259,2185
1,74-78,0.902367,5787
2,76-80,0.911181,7656
3,78-82,0.918210,7666
4,80-84,0.920800,6048
5,82-86,0.920413,4649
6,84-88,0.923683,3551
7,86-90,0.927095,2661
8,88-92,0.924382,2063
9,90-94,0.924653,1659
